# Auditing Content Moderation AI for Bias, Adversarial Robustness & Safety

End-to-end audit of a DistilBERT toxicity classifier on the Jigsaw Unintended Bias dataset.

## Section 1 — Setup and Data Loading

- Load 100k training rows + 20k evaluation rows (stratified on binarized `toxic`)
- Binarize `toxic` at threshold ≥ 0.5
- Inspect class balance and identity-column null summary

In [1]:
!pip install -q transformers torch scikit-learn fairlearn aif360 pandas matplotlib seaborn

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

USE_COLS = ["comment_text", "toxic", "black", "white", "muslim", "jewish", "homosexual_gay_or_lesbian"]
IDENTITY_COLS = ["black", "white", "muslim", "jewish", "homosexual_gay_or_lesbian"]

raw = pd.read_csv("data/jigsaw-unintended-bias-train.csv", usecols=USE_COLS)
raw["label"] = (raw["toxic"] >= 0.5).astype(int)

# Stratified 100k train / 20k eval split on the binarized label.
sample = raw.sample(n=120_000, random_state=SEED)
train_df, eval_df = train_test_split(
    sample, test_size=20_000, stratify=sample["label"], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

print("train:", train_df.shape, "eval:", eval_df.shape)
print("\nClass balance:")
print(pd.concat([
    train_df["label"].value_counts(normalize=True).rename("train"),
    eval_df["label"].value_counts(normalize=True).rename("eval"),
], axis=1).round(4))

print("\nIdentity null %:")
print(train_df[IDENTITY_COLS].isna().mean().round(4))

train: (100000, 8) eval: (20000, 8)

Class balance:
        train    eval
label                
0      0.9202  0.9202
1      0.0798  0.0798

Identity null %:
black                        0.7751
white                        0.7751
muslim                       0.7751
jewish                       0.7751
homosexual_gay_or_lesbian    0.7751
dtype: float64


## Section 2 — Baseline DistilBERT Fine-Tuning

- Tokenize `comment_text` with `distilbert-base-uncased` (max_length=128)
- Fine-tune for 3 epochs with the HuggingFace `Trainer`
- Report Accuracy, Macro-F1, AUC-ROC on the 20k eval set
- Save the checkpoint to `checkpoints/baseline/` for reuse in later sections

In [3]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128
CKPT_DIR = "checkpoints/baseline"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ToxicDataset(Dataset):
    def __init__(self, df):
        enc = tokenizer(
            df["comment_text"].astype(str).tolist(),
            truncation=True, padding="max_length", max_length=MAX_LEN,
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            "input_ids": torch.tensor(self.input_ids[i]),
            "attention_mask": torch.tensor(self.attention_mask[i]),
            "labels": torch.tensor(self.labels[i]),
        }

train_ds = ToxicDataset(train_df)
eval_ds = ToxicDataset(eval_df)
print("tokenized:", len(train_ds), "train /", len(eval_ds), "eval")

/Users/ammarkashif/Documents/Code/res-assignment-2/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenized: 100000 train / 20000 eval


In [4]:
def compute_metrics(pred):
    logits, labels = pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "auc_roc": roc_auc_score(labels, probs),
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=200,
    seed=SEED,
    report_to="none",
    fp16=torch.backends.mps.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
metrics = trainer.evaluate()
print({k: round(v, 4) for k, v in metrics.items() if k.startswith("eval_")})

trainer.save_model(CKPT_DIR)
tokenizer.save_pretrained(CKPT_DIR)
print(f"saved to {CKPT_DIR}")